In [ ]:
import pandas as pd

file_path = "../data/wdi_fertility_data_filled_filtered.csv"
data = pd.read_csv(file_path)

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 310 entries, 0 to 309
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   year                        310 non-null    int64  
 1   country_name                310 non-null    object 
 2   gdp_per_capita_2015_dollar  310 non-null    float64
 3   physicians_per_1000         310 non-null    float64
 4   hospital_beds_per_1000      310 non-null    float64
 5   crop_production_index       310 non-null    float64
 6   life_expectancy_at_birth    310 non-null    float64
 7   total_fertility_rate        310 non-null    float64
dtypes: float64(6), int64(1), object(1)
memory usage: 19.5+ KB


In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Copy of data for transformations
data_prepared = data.copy()

data_prepared = data_prepared.dropna()

# Step 1: Encode the categorical variable 'country_name'
label_encoder = LabelEncoder()
data_prepared["country_id"] = label_encoder.fit_transform(data_prepared["country_name"])

# Step 2: Normalize numerical features (except 'year', 'country_id', and 'total_fertility_rate')
scaler = StandardScaler()
numerical_columns = [
    "gdp_per_capita_2015_dollar",
    "physicians_per_1000",
    "hospital_beds_per_1000",
    "crop_production_index",
    "life_expectancy_at_birth",
]

data_prepared[numerical_columns] = scaler.fit_transform(
    data_prepared[numerical_columns]
)

# Step 3: Create train, validation, and test splits
# Define train (1990-2015), validation (2016-2020), and test (2021-2025) splits
data_prepared["split"] = np.where(
    data_prepared["year"] <= 2015,
    "train",
    np.where(data_prepared["year"] <= 2020, "val", "test"),
)

# Create a complete sequence of years for each group
data_prepared = data_prepared.set_index(["country_id", "year"]).sort_index()

# Fill in missing years for each country with NaNs
data_prepared = data_prepared.reindex(
    pd.MultiIndex.from_product(
        [
            data_prepared.index.get_level_values("country_id").unique(),
            range(
                data_prepared.index.get_level_values("year").min(),
                data_prepared.index.get_level_values("year").max() + 1,
            ),
        ],
        names=["country_id", "year"],
    )
).reset_index()

# Forward fill or interpolate missing values
data_prepared[numerical_columns] = data_prepared[
    numerical_columns
].interpolate()  # Linear interpolation
data_prepared["total_fertility_rate"] = data_prepared[
    "total_fertility_rate"
].interpolate()

data_prepared["time_idx"] = data_prepared.groupby("country_id").cumcount()

data_prepared = data_prepared.drop(columns=["year", "country_name"])

data_prepared.head(), data_prepared["split"].value_counts()

(   country_id  gdp_per_capita_2015_dollar  physicians_per_1000  \
 0           0                   -0.639423             0.531352   
 1           0                   -0.598770             0.516587   
 2           0                   -0.561601             0.501822   
 3           0                   -0.520210             0.552164   
 4           0                   -0.491266             0.602507   
 
    hospital_beds_per_1000  crop_production_index  life_expectancy_at_birth  \
 0               -0.168094              -2.262951                 -0.036115   
 1               -0.167608              -2.113435                  0.016311   
 2               -0.167122              -2.079841                  0.027188   
 3               -0.166636              -2.185354                  0.040417   
 4               -0.278342              -2.039623                  0.099899   
 
    total_fertility_rate  split  time_idx  
 0                 3.034  train         0  
 1                 3.012  train 

In [66]:
from pytorch_forecasting.data import TimeSeriesDataSet
from pytorch_forecasting.models.temporal_fusion_transformer import (
    TemporalFusionTransformer,
)
from pytorch_forecasting.metrics import QuantileLoss
from pytorch_lightning import Trainer
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch

# Step 1: Define the PyTorch Forecasting TimeSeriesDataSet
# Identify max prediction length (forecast horizon) and max encoder length (lookback window)
max_prediction_length = 5  # Forecast next 5 years
max_encoder_length = 15  # Use up to 30 years of historical data

# Define TimeSeriesDataSet for train and validation
training_cutoff = data_prepared[data_prepared["split"] == "train"]["time_idx"].max()

data_prepared["country_id"] = data_prepared["country_id"].astype(str)


def custom_collate_fn(batch):
    x_cat = pad_sequence([b["x_cat"] for b in batch], batch_first=True)
    x_cont = pad_sequence([b["x_cont"] for b in batch], batch_first=True)
    return {"x_cat": x_cat, "x_cont": x_cont}


train_data = TimeSeriesDataSet(
    data_prepared[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="total_fertility_rate",
    group_ids=["country_id"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=["country_id"],
    time_varying_known_reals=["time_idx"] + numerical_columns,
    time_varying_unknown_reals=["total_fertility_rate"],
    target_normalizer=None,
)

validation_data = TimeSeriesDataSet.from_dataset(
    train_data, data_prepared, min_prediction_idx=training_cutoff + 1
)

# Step 2: Create DataLoaders
batch_size = 16  # Adjust based on GPU memory
train_dataloader = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True,
)
val_dataloader = DataLoader(
    validation_data,
    batch_size=batch_size,
)

# Step 3: Define Temporal Fusion Transformer Model
tft = TemporalFusionTransformer.from_dataset(
    train_data,
    learning_rate=0.03,
    hidden_size=16,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=8,
    output_size=7,  # Quantiles for prediction intervals
    loss=QuantileLoss(),  # Replace with quantile loss if prediction intervals are needed
    log_interval=10,
    reduce_on_plateau_patience=4,
)

# Step 4: Train the Model
trainer = Trainer(
    accelerator="cpu",
    max_epochs=30,  # Adjust based on validation performance
    gradient_clip_val=0.1,
)

# Train the model
trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

# Save the trained model
model_path = "../data/tft_fertility_model.pth"
torch.save(tft.state_dict(), model_path)

model_path

/Users/kacperkedzierski/Documents/Code/Repositories/fertility-prognosing/.venv/lib/python3.11/site-packages/pytorch_lightning/utilities/parsing.py:209: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/Users/kacperkedzierski/Documents/Code/Repositories/fertility-prognosing/.venv/lib/python3.11/site-packages/pytorch_lightning/utilities/parsing.py:209: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
/Users/kacperkedzierski/Documents/Code/Repositories/fertility-prognosing/.venv/lib/python3.11/site-packages/pytorch_forecasting/models/temporal_fusion_transformer/__init__.py:171: UserWarning: In pytorch-forecasting models, on versions 1.1.X, the default optimizer defaults to 'adam', if pytorch_optimizer is not installed, 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/kacperkedzierski/Documents/Code/Repositories/fertility-prognosing/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'NoneType'>

In [35]:
from sklearn.preprocessing import StandardScaler

# Check for missing or invalid values in numerical columns
numerical_check = validation_data[numerical_columns].isnull().sum()
print("Missing values in numerical columns:", numerical_check)

# Manually inspect scaling
scaler = StandardScaler()
scaled_features = scaler.fit_transform(validation_data[numerical_columns])
print("Scaled features shape:", scaled_features.shape)

Missing values in numerical columns: gdp_per_capita_2015_dollar    0
physicians_per_1000           0
hospital_beds_per_1000        0
crop_production_index         0
life_expectancy_at_birth      0
dtype: int64
Scaled features shape: (50, 5)
